In [79]:
# load the module
%load_ext autoreload
%autoreload 2

import sys 
sys.path.append("./Qtensor")
sys.path.append("./Qtensor/qtree_git")
sys.path.append("./Qtensor/qtree_git/qtree")
sys.path.append("./classical_benchmarks")
sys.path.append("./Experiments/SetCover")
sys.path.append("./Experiments/MAXCUT_RQAOA/scripts/")
sys.path.append(".")
sys.path.append("./Experiments/SetCover/scripts/")
sys.path.append("./Helping_tools")

import torch
import Generating_Problems as Generator
from Calculating_Expectation_Values import SingleLayerQAOAExpectationValues, QtensorQAOAExpectationValuesQUBO
from QIRO import QIRO_MIS, QIRO_SetCover
from RQAOA import RQAOA_recalculate
import networkx as nx
import numpy as np
from greedy_mis import min_greedy_mis, max_greedy_mis
from greedy_set_cover import greedy_set_cover
from Create_instances import create_set_cover


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Run QIRO on MIS problems

1. Create networkx graphs for corresponding MIS problems.

2. Generate the corresponding problem class.

3. Specify which QIRO variation you want to use from {'MAXQ', 'MINQ', 'MMQ' and the original 'QIRO'}.

Specify which parameter initialization for QAOA is chosen out of {'interpolation', 'transition_states', 'random'}.

Specify depth p of QAOA. 

Optionally specify which optimizer and learning rates are used in QAOA parameter optimization.  

Generate expectation values class for calculating correlations within future shrinking algorithms. 

4. Specify which size nq the subgraphs within the problem have that are solved clasically. 

Generate QIRO class and execute it. 

5. Extract results. 

6. Solve problem greedily.

1. 

Create networkx graphs for corresponding MIS problems.

In [41]:
regularity = 3
n = 10
seed = 300

# create random regular graph
G = nx.random_regular_graph(regularity, n, seed=seed)

#for Erdos Renyi graphs:
#prob = reg/(n-1) 
#G = nx.erdos_renyi_graph(n, prob)

2. 

Generate the corresponding MIS problem class.

In [42]:
# generate MIS problem instance
problem = Generator.MIS(G, alpha = 1.1)

3. 

Specify which QIRO variation you want to use from {'MAXQ', 'MINQ', 'MMQ' and the original 'QIRO'}.

Specify which parameter initialization for QAOA is chosen out of {'interpolation', 'transition_states', 'random'}.

Specify depth p of QAOA. 

Optionally specify which optimizer and learning rates are used in QAOA parameter optimization.  

Generate expectation values class for calculating correlations within future shrinking algorithms. 


In [43]:
# generate correlations for shrinking
variation = 'MINQ'
initialization = 'interpolation'
p = 2

expectation_values_qtensor = QtensorQAOAExpectationValuesQUBO(problem, p, opt=torch.optim.RMSprop, variation = variation, initialization = initialization, opt_kwargs=dict(lr=0.005))


4. 

Specify which size nq the subgraphs within the problem have that are solved clasically. 

Generate QIRO class and execute it. 

In [44]:
nq = 5

QIRO_TN = QIRO_MIS(nq, expectation_values_qtensor, variation=variation)

QIRO_TN.execute()

Step: 1. Number of nodes: 10.
stufe von interpolationt p = 2


 68%|██████▊   | 34/50 [00:21<00:09,  1.61it/s]

single var [2]. Sign: 1
Pruned 6 variables.
We have fixed the following variables: [[3], [9], [8], [2], [1], [4], [5], [6], [7], [10]]. Moving on.
Solution: [-1, 2, -3, 4, 5, -6, -7, -8, -9, 10]


5. 

Extract results.

In [45]:
# get solution
solution = QIRO_TN.solution
# get size of independent set
size_indep_set_qiro_TN = np.sum(solution > 0) 
# get list of QAOA energies of individual shrinking steps 
energies = QIRO_TN.energies_list
# get list of losses during QAOA optimization within each shrinking step
losses = QIRO_TN.losses_list
# get list with number of nodes for each shrinking step
num_nodes = QIRO_TN.num_nodes
# get list with all correlations in each shrinking step
correlations = QIRO_TN.correlations

6. 

Solve problem greedily.

In [46]:
# using MIN greedy algorithm
size_min_greedy = min_greedy_mis(G)
# using MAX greedy algorithm
size_max_greedy = max_greedy_mis(G)

## Run QIRO on SetCover problems

1. Create SetCover problem mimicking sensor positioning problem on a grid of size a x b.

2. Generate SetCover problem class. Check on number of required qubits.

3. Specify which QIRO variation you want to use from {'MAXQ', 'MINQ', 'MMQ'}.

Specify which parameter initialization for QAOA is chosen out of {'interpolation', 'transition_states', 'random'}.

Specify depth p of QAOA. 

Optionally specify which optimizer and learning rates are used in QAOA parameter optimization.  

Generate expectation values class for calculating correlations within future shrinking algorithms. 

4. Specify which size nq the subgraphs within the problem have that are solved clasically. 

Generate QIRO class, execute it and receive results. More information of shrinking steps can be obtained the same way as for MIS problems above. 

5. Optionally do a validity check of the solution.Extract results. 

6. Solve problem greedily.

1. 

Create SetCover problem mimicking sensor positioning problem on a grid of size a x b.

In [47]:
a = 3
b = 5

U, V = create_set_cover(x=a, y=b)

2. 
Generate SetCover problem class. Check on number of required qubits.

In [48]:
problem = Generator.SetCover(U, V, A=2, B=1)  
num_qubits = problem.num_variables


3. 
Specify which QIRO variation you want to use from {'MAXQ', 'MINQ', 'MMQ'}.

Specify which parameter initialization for QAOA is chosen out of {'interpolation', 'transition_states', 'random'}.

Specify depth p of QAOA. 

Optionally specify which optimizer and learning rates are used in QAOA parameter optimization.  

Generate expectation values class for calculating correlations within future shrinking algorithms. 

In [49]:
variation = 'MINQ'
initialization = 'interpolation'
p = 1
expectation_value_TN = QtensorQAOAExpectationValuesQUBO(problem, p=p, variation=variation, opt=torch.optim.RMSprop, initialization = 'interpolation', opt_kwargs=dict(lr=0.001))


4. 

Specify which size nq the subgraphs within the problem have that are solved clasically. 

Generate QIRO class, execute it and receive results. More information of shrinking steps can be obtained the same way as for MIS problems above. 

In [50]:
nq = 1
MMQ = QIRO_SetCover(nq, expectation_value_TN, variation=variation)
shrinking_solution, shrinking_size = MMQ.execute()     

Step: 1. Number of nodes: 42.


/Users/Q642074/miniconda3/envs/qiro/lib/python3.9/site-packages/scipy/optimize/_minpack_py.py:177: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)
/Users/Q642074/miniconda3/envs/qiro/lib/python3.9/site-packages/scipy/optimize/_minpack_py.py:177: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last five Jacobian evaluations.
  warnings.warn(msg, RuntimeWarning)


9
single var [1]. Sign: 1
chosen original [1]
b [[1], [6], [4, 5], [4, 5], [2, 8], [0, 5, 7], [0, 1, 9], [3], [10]]
subset [1]
a [[6], [4, 5], [4, 5], [2, 8], [0, 5, 7], [0, 9], [3], [10]]
Pruned 0 variables.
We have fixed the following variables: [[1]]. Moving on.
Step: 2. Number of nodes: 38.
8
single var [2]. Sign: 1
chosen original [4, 5]
b [[6], [4, 5], [4, 5], [2, 8], [0, 5, 7], [0, 9], [3], [10]]
subset [4, 5]
a [[6], [], [2, 8], [0, 7], [0, 9], [3], [10]]
original [[6], [4, 5], [2, 8], [0, 5, 7], [0, 1, 9], [3], [10]]
Pruned 0 variables.
We have fixed the following variables: [[], [4, 5]]. Moving on.
Step: 3. Number of nodes: 22.
6
single var [2]. Sign: 1
chosen original [2, 8]
b [[6], [2, 8], [0, 7], [0, 9], [3], [10]]
subset [2, 8]
a [[6], [0, 7], [0, 9], [3], [10]]
Pruned 0 variables.
We have fixed the following variables: [[2, 8]]. Moving on.
Step: 4. Number of nodes: 17.
5
single var [2]. Sign: 1
chosen original [0, 5, 7]
b [[6], [0, 7], [0, 9], [3], [10]]
subset [0, 7]
a 

5. 
Optionally do a validity check of the solution.

In [51]:
valid, rest_size = problem.solution_check(list(shrinking_solution))


6. 
Solve problem greedily.

In [52]:
greedy_solution, greedy_size = greedy_set_cover(U, V)


## Run RQAOA with recalculations on MaxCut problems

1. Generate networkx graph of a MaxCut problem.

2. Generate the corresponding MaxCut problem class.

3. Specify which parameter initialization for QAOA is chosen out of {'fixed_angles', 'fixed_angles_optimization', 'interpolation', 'transition_states', 'random'}.

Specify depth p of QAOA. 

Optionally specify which optimizer and learning rates are used in QAOA parameter optimization.  

Generate expectation values class for calculating correlations within future shrinking algorithms.

4. Specify which size nq the subgraphs within the problem have that are solved clasically.

Specify the recalculation interval r in which no new correlations are calculated. 

Generate RQAOA class and execute it. 

5. 
Extract results.

1. 
Generate networkx graph of a MaxCut problem.

In [75]:
regularity = 3
n = 12
seed = 300

# create random regular graph
G = nx.random_regular_graph(regularity, n, seed=seed)

#for Erdos Renyi graphs:
#prob = reg/(n-1) 
#G = nx.erdos_renyi_graph(n, prob)


2. 

Generate the corresponding MaxCut problem class.

In [76]:
problem = Generator.MAXCUT(G)


3. 

Specify which parameter initialization for QAOA is chosen out of {'fixed_angles', 'fixed_angles_optimization', 'interpolation', 'transition_states', 'random'}.

Specify depth p of QAOA. 

Optionally specify which optimizer and learning rates are used in QAOA parameter optimization.  

Generate expectation values class for calculating correlations within future shrinking algorithms. 

In [81]:
initialization = 'fixed_angles_optimization'
p = 2

expectation_values_qtensor = QtensorQAOAExpectationValuesQUBO(problem, p, initialization='fixed_angles_optimization', opt=torch.optim.SGD, opt_kwargs=dict(lr=0.0001))


4. 

Specify which size nq the subgraphs within the problem have that are solved clasically.

Specify the recalculation interval r in which no new correlations are calculated. 

Generate RQAOA class and execute it. 

In [98]:
nq = 5
r = 5

RQAOA_TN = RQAOA_recalculate(expectation_values_qtensor, nq, recalculations=r, type_of_problem="MAXCUT")

cuts_TN, solution_TN = RQAOA_TN.execute()



RQAOA Step: 1


 14%|█▍        | 7/50 [00:02<00:17,  2.41it/s]


RQAOA Step: 2
RQAOA Step: 3
RQAOA Step: 4
RQAOA Step: 5
RQAOA Step: 6


 50%|█████     | 25/50 [00:05<00:05,  4.17it/s]

RQAOA Step: 7
16 [ -1  -2   3   4  -5  -6   7   8  -9 -10  11  12]


5. 
Extract results.

In [ ]:
# get number of cuts
num_cuts = cuts_TN
# get solution
solution =  solution_TN
# get list of QAOA energies of individual shrinking steps 
energies =  RQAOA_TN.energies_list
# get list of losses during QAOA optimization within each shrinking step
losses = RQAOA_TN.losses_list
# get list with number of nodes for each shrinking step
num_nodes = RQAOA_TN.num_nodes_list
# get list with connectivity information for each shrinking step
connectivities = RQAOA_TN.connectivity
# get list with all correlations in each shrinking step
correlations = RQAOA_TN.correlations
    

## Further information

If you want to use only p=1 QAOA for obtaining correlations, you can do this without tensor networks but only using analytic calculations. Just replace "QtensorQAOAExpectationValuesQUBO(problem, ...)" with "SingleLayerQAOAExpectationValues(problem)" in the above examples.